# Audit stratifikasi tersembunyi: pengelompokan tanpa label foto e-waste Elektronik

Notebook ini menjalankan langkah yang sama dengan `run_all.py`, satu per satu, dengan penjelasan singkat.

* Input: embedding DINOv3 ViT-B/16 (144 px, 768 dimensi, tanpa fine-tuning) untuk 3.607 foto Elektronik yang tersisa setelah deduplikasi.
* Semua langkah pengelompokan dan pemilihan konfigurasi tidak memakai label apa pun.
* Label manusia hanya untuk evaluasi dan tidak dibaca di notebook ini.
* Nama dari nama file (10 jenis perangkat, hanya ada untuk foto katalog) baru dipakai di tahap audit, setelah pilihan dibuat.

Notebook memakai profil `quick` (6 kandidat cepat, sekitar 5 menit). Profil `full` di `run_all.py` menambah 4 kandidat yang lambat.

In [1]:
import os
import sys
from pathlib import Path

PKG = Path.cwd()
sys.path.insert(0, str(PKG))
from src.utils import limit_threads

limit_threads(1)

import io
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import adjusted_rand_score

from src import data, pipeline, utils

warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm without ipywidgets
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
cfg = utils.load_config(PKG / "config.yaml")
ROOT = os.environ.get("SD_ROOT")  # isi path folder SD/ di sini jika variabel lingkungan tidak ada
paths = utils.resolve_paths(cfg, ROOT)
log = utils.get_logger()
print("root:", paths["root"])
print({k: v for k, v in utils.library_versions().items() if k in ("python", "numpy", "scikit-learn", "scanpy", "igraph")})

root: /mnt/user-data/uploads/SD
{'python': '3.11.15', 'numpy': '2.4.4', 'scikit-learn': '1.8.0', 'scanpy': '1.11.5', 'igraph': '1.0.0'}


## 1. Data

Baris yang dipakai adalah kelas `1_Electronic` dengan `keep == 1`, yaitu satu foto per kelompok duplikat (perceptual hash). Hasilnya 3.607 foto, terdiri dari 2.332 foto katalog dan 1.275 foto lapangan (scraped).

In [2]:
ds = data.load_dataset(paths["embeddings"], paths["index"], cfg)
print("embedding:", ds.E.shape, ds.E.dtype)
ds.index.provenance.value_counts().rename("jumlah foto").to_frame()

embedding: (3607, 768) float64


,jumlah foto
provenance,
catalogue,2332
scraped,1275


## 2. Fitur

Fitur utama (`raw`) adalah embedding yang dikurangi rata-ratanya lalu dinormalisasi L2 per baris. Varian `defl20` membuang 20 komponen utama terbesar sebelum normalisasi. Varian ini hanya menjadi kandidat pembanding.

In [3]:
features = data.build_features(ds.E, cfg)
{name: (Z.shape, round(float(np.linalg.norm(Z, axis=1).mean()), 6)) for name, Z in features.items()}

{'raw': ((3607, 768), 1.0), 'defl20': ((3607, 768), 1.0)}

## 3. Kandidat dan validitas internal

Setiap kandidat dijalankan sekali pada semua foto, lalu diulang pada 20 subsampel 80% dan dengan 5 seed lain.

* Stabilitas resampling adalah rata-rata ARI antarpasangan subsampel, dihitung pada foto yang sama-sama ada di kedua subsampel.
* Stabilitas seed adalah rata-rata ARI antarpasangan dari 6 hasil (seed 0 sampai 5).
* Indeks internal adalah silhouette kosinus, Davies-Bouldin dan Calinski-Harabasz.

Angka TEMI disalin dari run asli (tidak dilatih ulang) dan diberi keterangan pada kolom `source`.

In [4]:
cand = pipeline.evaluate_candidates(features, cfg, "quick", data_id="notebook", cache_dir=None, logger=log)
cand.table.round(3)

19:44:28  raw__KMeans_k10        k= 10 sil=0.200 resample ARI=0.899 seed ARI=0.930 (31 s)


19:45:11  raw__KMeans_k20        k= 20 sil=0.208 resample ARI=0.836 seed ARI=0.823 (44 s)


19:45:34  raw__Leiden_r0.5       k= 17 sil=0.212 resample ARI=0.890 seed ARI=0.950 (23 s)


19:45:43  raw__Leiden_r1.0       k= 19 sil=0.213 resample ARI=0.902 seed ARI=0.948 (8 s)


19:45:51  defl20__Leiden_r0.5    k= 13 sil=-0.016 resample ARI=0.708 seed ARI=0.911 (9 s)


19:45:59  defl20__Leiden_r1.0    k= 24 sil=-0.010 resample ARI=0.711 seed ARI=0.891 (8 s)


19:45:59  not run in the 'quick' profile: raw__UMAP_HDBSCAN, defl20__KMeans_k10, defl20__KMeans_k20, defl20__UMAP_HDBSCAN


,key,features,method,source,n_clusters,noise_frac,largest_frac,sil,db,ch,stab_sub_ari,stab_sub_sd,n_sub,stab_seed_ari,stab_seed_sd,n_seed_labellings
0,raw__KMeans_k10,raw,KMeans k=10,computed,10,0.0,0.203,0.200,2.741,151.085,0.899,0.053,20,0.930,0.051,6
1,raw__KMeans_k20,raw,KMeans k=20,computed,20,0.0,0.074,0.208,2.966,105.752,0.836,0.046,20,0.823,0.039,6
2,raw__Leiden_r0.5,raw,Leiden r=0.5,computed,17,0.0,0.105,0.212,2.431,107.688,0.890,0.028,20,0.950,0.016,6
3,raw__Leiden_r1.0,raw,Leiden r=1.0,computed,19,0.0,0.106,0.213,2.422,102.310,0.902,0.038,20,0.948,0.023,6
4,defl20__Leiden_r0.5,defl20,Leiden r=0.5,computed,13,0.0,0.169,-0.016,13.756,5.725,0.708,0.064,20,0.911,0.033,6
5,defl20__Leiden_r1.0,defl20,Leiden r=1.0,computed,24,0.0,0.092,-0.010,8.872,7.364,0.711,0.046,20,0.891,0.027,6
6,raw__TEMI_K10,raw,TEMI K=10,"recorded (original TEMI run, not re-fitted)",10,0.0,0.141,0.169,2.963,124.394,0.532,NaN,3,0.481,NaN,16
7,raw__TEMI_K20,raw,TEMI K=20,"recorded (original TEMI run, not re-fitted)",20,0.0,0.097,0.190,3.047,94.243,NaN,NaN,0,0.703,NaN,16
8,defl20__TEMI_K10,defl20,TEMI K=10,"recorded (original TEMI run, not re-fitted)",10,0.0,0.182,0.004,8.694,11.574,NaN,NaN,0,0.094,NaN,16


## 4. Pilihan tanpa label

Aturan ditulis sebelum evaluasi. Dari konfigurasi dengan paling sedikit 10 klaster, pilih stabilitas resampling tertinggi. Bila selisihnya dengan yang lain kurang dari 0,01, pilih silhouette yang lebih tinggi.

In [5]:
key, table, md = pipeline.run_selection(cand.table, cfg, cand.skipped)
display(Markdown(md))

# Label-free choice of the configuration

Only label-free columns were read.

**Rule:** Among all configurations with >= 10 non-noise clusters and a resampling-stability estimate, choose the one with the highest resampling stability (mean pairwise ARI over 80% subsamples, compared on shared points). If two are within 0.01 of each other, prefer the higher cosine silhouette. Only label-free quantities are read.

**Chosen:** `raw__Leiden_r1.0` (raw, Leiden r=1.0): 19 clusters, resampling ARI 0.902, seed ARI 0.948, cosine silhouette 0.213.

Eligible candidates, sorted by resampling stability:

| key                 | source                                      |   n_clusters |   stab_sub_ari |   n_sub |   stab_seed_ari |    sil |
|:--------------------|:--------------------------------------------|-------------:|---------------:|--------:|----------------:|-------:|
| raw__Leiden_r1.0    | computed                                    |           19 |          0.902 |      20 |           0.948 |  0.213 |
| raw__KMeans_k10     | computed                                    |           10 |          0.899 |      20 |           0.93  |  0.2   |
| raw__Leiden_r0.5    | computed                                    |           17 |          0.89  |      20 |           0.95  |  0.212 |
| raw__KMeans_k20     | computed                                    |           20 |          0.836 |      20 |           0.823 |  0.208 |
| defl20__Leiden_r1.0 | computed                                    |           24 |          0.711 |      20 |           0.891 | -0.01  |
| defl20__Leiden_r0.5 | computed                                    |           13 |          0.708 |      20 |           0.911 | -0.016 |
| raw__TEMI_K10       | recorded (original TEMI run, not re-fitted) |           10 |          0.532 |       3 |           0.481 |  0.169 |

Not eligible (fewer than 10 clusters or no resampling estimate): raw__TEMI_K20, defl20__TEMI_K10.

Not run in this profile (use --profile full): raw__UMAP_HDBSCAN, defl20__KMeans_k10, defl20__KMeans_k20, defl20__UMAP_HDBSCAN.


## 5. Cek reproduksi

Label hasil notebook dibandingkan dengan label Leiden r=1,0 dari analisis asli (`reference/raw__Leiden_r1.0.npy`).

In [6]:
labels = cand.refits[key].labels
ref = np.load(paths["reference"] / "raw__Leiden_r1.0.npy")
print("konfigurasi terpilih:", key, "| sama dengan pilihan asli:", key == cfg["selection"]["expected_choice"])
print("ARI terhadap label asli:", adjusted_rand_score(ref, labels), "| identik:", np.array_equal(ref, labels))
pd.Series(labels).value_counts().sort_index().rename("ukuran").to_frame().T

konfigurasi terpilih: raw__Leiden_r1.0 | sama dengan pilihan asli: True
ARI terhadap label asli: 1.0 | identik: True


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
ukuran,281,384,148,361,58,228,220,119,192,257,268,204,147,249,288,108,21,38,36


## 6. Audit dengan nama file

Tahap ini berjalan setelah pilihan dibuat.

* Tabel silang klaster terhadap nama file hanya memakai foto katalog.
* Foto lapangan diberi nama lewat voting 15 tetangga katalog terdekat. Nama dianggap yakin bila minimal 60% tetangga setuju.
* Skor pecah sebuah nama adalah entropi sebaran fotonya di antara klaster, dibagi ln(jumlah klaster). Nilai 0 berarti satu klaster.
* Klaster tanpa nama adalah klaster yang nama mayoritas yakinnya menutup kurang dari 50% anggotanya.

In [7]:
aud = pipeline.run_audit(ds, labels, cand.refits[key], cand.subsamples, cfg)
aud.crosstab_catalogue.set_index("cluster")

name,Keyboard,Microwave,Mobile,Mouse,Player,Printer,Television,Washing_Machine,battery,pcb
cluster,,,,,,,,,,
0,0,0,0,0,0,3,1,0,0,1
1,1,0,0,1,0,1,0,0,0,215
2,9,0,0,0,0,0,0,0,0,0
3,3,0,145,1,0,0,1,0,0,0
4,0,0,0,0,0,0,0,0,0,0
5,0,1,0,0,226,0,0,0,0,0
6,0,5,0,0,1,0,123,0,0,0
7,2,0,0,0,0,0,1,0,0,0
8,11,1,0,0,2,0,0,0,177,0


In [8]:
aud.split_scores.round(3)

,view,name,n,split_score,effective_clusters,top_cluster,top_share,n_parts,parts,split,majority_in_clusters
0,catalogue filename name,Keyboard,233,0.194,1.770,9,0.876,1,9:0.876,False,2 9
1,catalogue filename name,Microwave,252,0.086,1.287,13,0.956,1,13:0.956,False,13
2,catalogue filename name,Mobile,150,0.060,1.195,3,0.967,1,3:0.967,False,3 4 16
3,catalogue filename name,Mouse,206,0.021,1.063,14,0.990,1,14:0.990,False,14 17
4,catalogue filename name,Player,237,0.093,1.315,5,0.954,1,5:0.954,False,5
5,catalogue filename name,Printer,275,0.082,1.273,10,0.956,1,10:0.956,False,10
6,catalogue filename name,Television,254,0.267,2.192,12,0.500,2,12:0.500 6:0.484,True,6 7 12
7,catalogue filename name,Washing_Machine,294,0.250,2.088,11,0.643,2,11:0.643 15:0.340,True,11 15
8,catalogue filename name,battery,214,0.164,1.619,8,0.827,2,8:0.827 18:0.168,True,8 18
9,catalogue filename name,pcb,217,0.020,1.061,1,0.991,1,1:0.991,False,0 1


In [9]:
aud.orphans.round(3)

,cluster,size,n_catalogue,catalogue_share,n_named,majority_name,majority_n,majority_share,majority_share_of_named,second_name,second_share,orphan
0,0,281,5,0.018,161,pcb,89,0.317,0.553,Printer,0.075,True
1,1,384,218,0.568,374,pcb,364,0.948,0.973,Printer,0.008,False
2,2,148,9,0.061,135,Keyboard,108,0.730,0.800,Mobile,0.108,False
3,3,361,150,0.416,357,Mobile,350,0.970,0.980,Keyboard,0.008,False
4,4,58,0,0.000,41,Mobile,35,0.603,0.854,Keyboard,0.052,False
5,5,228,227,0.996,228,Player,227,0.996,0.996,Microwave,0.004,False
6,6,220,129,0.586,218,Television,207,0.941,0.950,Microwave,0.023,False
7,7,119,3,0.025,108,Television,83,0.697,0.769,Keyboard,0.151,False
8,8,192,191,0.995,192,battery,178,0.927,0.927,Keyboard,0.057,False
9,9,257,205,0.798,257,Keyboard,256,0.996,0.996,Player,0.004,False


## 7. Stabilitas pecahan dan klaster tanpa nama

Untuk tiap klaster acuan dihitung bagian anggotanya yang jatuh ke satu klaster yang sama pada 25 pengulangan (20 subsampel dan 5 seed). Untuk tiap pasangan dihitung seberapa sering keduanya tetap terpisah.

In [10]:
display(aud.stability.round(3))
aud.stability_pairs

,cluster,group,n,mean_top_share,min_top_share,frac_refits_hold,n_refits
0,0,heaps 0,281,0.976,0.947,1.00,25
1,1,,384,0.967,0.604,1.00,25
2,2,laptop 2,148,0.967,0.888,1.00,25
3,3,smartphone 3,361,0.978,0.926,1.00,25
4,4,laptop 4,58,0.968,0.833,1.00,25
5,5,,228,0.754,0.448,0.64,25
6,6,CRT 6,220,0.783,0.640,1.00,25
7,7,,119,0.841,0.663,1.00,25
8,8,small batt 8,192,0.852,0.573,0.96,25
9,9,,257,0.974,0.762,1.00,25


,pair,cluster_a,cluster_b,frac_refits_apart,n_refits
0,CRT 6 vs flat TV 12,6,12,1.00,25
1,small batt 8 vs aki 18,8,18,0.92,25
2,smartphone 3 vs feature phone 16,3,16,1.00,25


## 8. Ekspor untuk gambar

Tabel per foto berisi klaster, sumber foto, nama file, nama hasil voting dan posisi ForceAtlas2 dari graf kNN mentah. Gambar dibuat oleh skrip terpisah. Sel ini juga memeriksa apakah tabel identik byte demi byte dengan `outputs/clusters.csv` bila file itu sudah ada.

In [11]:
clusters, pos_source = pipeline.run_export(ds, labels, aud, cfg, paths["reference"])
buf = io.StringIO()
clusters.to_csv(buf, index=False, float_format=cfg["output"]["float_format"], lineterminator="\n")
saved = paths["outputs"] / "clusters.csv"
if saved.exists():
    print("identik dengan outputs/clusters.csv:", buf.getvalue() == saved.read_text(encoding="utf-8"))
print("posisi FA2:", pos_source)
clusters.head()

identik dengan outputs/clusters.csv: True
posisi FA2: recomputed


,file,thumb,provenance,cluster,filename_name,pred_name,pred_conf,confident_name,fa2_x,fa2_y
0,1_E_1.jpg,1_E_1.jpg,scraped,0,,pcb,0.666667,pcb,921.561740,130.206176
1,1_E_2.jpg,1_E_2.jpg,scraped,1,,pcb,1.000000,pcb,1910.161658,-2442.286567
2,1_E_3.jpg,1_E_3.jpg,scraped,2,,Keyboard,0.933333,Keyboard,5447.239977,267.791593
3,1_E_4.jpg,1_E_4.jpg,scraped,0,,Washing_Machine,0.400000,,124.373785,525.316078
4,1_E_5.jpg,1_E_5.jpg,scraped,0,,pcb,1.000000,pcb,1031.904039,-961.059344


## Ringkasan

* Aturan tanpa label memilih Leiden r=1,0 pada graf kNN kosinus (k=15) dengan 19 klaster.
* Label hasil notebook sama persis dengan label analisis asli.
* Tabel audit menunjukkan nama yang terpecah ke beberapa klaster dan klaster yang tidak tertutup satu nama pun.
* Label manusia tidak dipakai di notebook ini.